# Lecture 9: Materials for Quantum and Molecular Photonics

## Overview
**Questions**
- What material platforms are used in quantum and molecular optics?
- How do we computationally characterise a quantum emitter?
- What are the key figures of merit for a solid-state quantum emitter?

**Objectives**
- Survey the main solid-state quantum emitter platforms
- Understand how DFT is used to characterise defect optical properties
- Explore the computational workflow for the NV centre in diamond and V_B in hBN
- Discuss emerging material platforms

---

> **Course context:** This lecture synthesises everything you have learned in Lectures 1–8 and applies it to the central research question: *How do we use atomistic simulation to understand and design solid-state quantum emitters?*


## Solid-State Single-Photon Emitters

A **single-photon emitter (SPE)** is a quantum system that, when excited, emits exactly one photon at a time. This is a fundamental resource for quantum communication, quantum computing, and quantum sensing.

### Figures of merit for an SPE:

| Property | Ideal value | Physical meaning |
|----------|-------------|-----------------|
| **Zero-phonon line (ZPL) fraction** | High (→1) | Fraction of emission into the coherent photon |
| **Linewidth** | Lifetime-limited | Coherence time = $T_2 = 2T_1$ |
| **Emission wavelength** | Telecom (1310/1550 nm) or visible | Compatibility with fibres or detectors |
| **Photon purity** $g^{(2)}(0)$ | <0.5 (→0) | Antibunching — proof of single-photon nature |
| **Brightness** | High count rate | Useful for applications |
| **Spin coherence** (for spin qubits) | Long $T_2$ | Needed for qubit applications |


## Platform 1: NV Centre in Diamond

The **nitrogen-vacancy (NV) centre** is the most studied solid-state qubit and SPE. It consists of a substitutional N atom adjacent to a carbon vacancy (V_C), forming a complex with $C_{3v}$ symmetry.

### Electronic structure

The NV⁻ has 6 electrons in the defect complex:
- Ground state: ${}^3A_2$ (spin triplet, $S=1$) — two degenerate $e$ orbitals occupied by one electron each
- Excited state: ${}^3E$ (spin triplet) — one electron promoted to a higher $e$ orbital
- ZPL energy: **1.945 eV (637 nm)**
- Spin coherence time $T_2$ up to **ms** at room temperature (world record for a solid-state qubit)

### Computational characterisation

Key quantities we compute with DFT:
1. **Formation energy** — thermodynamic stability of NV⁻ vs NV⁰
2. **Electronic level positions** — where do the defect states lie in the gap?
3. **ZPL energy** using $\Delta$SCF or constrained-occupation DFT
4. **Huang-Rhys factor** — phonon sideband structure
5. **Zero-field splitting** — for spin qubit applications


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuration Coordinate Diagram for the NV centre
# Schematic — real values from DFT + experimental literature

Q = np.linspace(-0.5, 2.5, 300)

# Parabolic potential wells (harmonic approximation)
# Ground state (³A₂)
E_gs = 0.5 * (Q - 0.0)**2

# Excited state (³E) — shifted by Delta_Q and raised by ZPL energy
ZPL = 1.945  # eV
Delta_Q = 1.0  # amu^0.5 Å (mass-weighted displacement)
E_es = ZPL + 0.5 * (Q - Delta_Q)**2

# Stokes shift and reorganisation energy
E_stokes = 0.5 * Delta_Q**2  # eV (approx)

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(Q, E_gs, 'b-', lw=2.5, label='Ground state $^3A_2$')
ax.plot(Q, E_es, 'r-', lw=2.5, label='Excited state $^3E$')

# ZPL arrow
ax.annotate('', xy=(0, ZPL), xytext=(0, 0),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2.5))
ax.text(0.07, ZPL/2, f'ZPL = {ZPL} eV
(637 nm)', color='green', fontsize=11)

# Absorption arrow (vertical from GS minimum)
E_abs = ZPL + E_stokes
ax.annotate('', xy=(0, E_abs), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2, ls='dashed'))
ax.text(-0.45, E_abs/2 + 0.3, 'Absorption', color='blue', fontsize=10, rotation=90)

# Emission arrow (vertical from ES minimum)
E_em = ZPL - E_stokes
ax.annotate('', xy=(Delta_Q, ZPL), xytext=(Delta_Q, E_em),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, ls='dashed'))
ax.text(Delta_Q + 0.07, (ZPL + E_em)/2, 'Emission', color='red', fontsize=10, rotation=90)

ax.set_xlabel('Configuration coordinate Q (amu$^{1/2}$Å)', fontsize=12)
ax.set_ylabel('Energy (eV)', fontsize=12)
ax.set_title('Configuration coordinate diagram — NV centre in diamond', fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim(-0.2, 3.5)
plt.tight_layout()
plt.show()

print(f"Stokes shift (approx): {E_stokes:.2f} eV")
print(f"Huang-Rhys factor S = E_reorg / ħω ≈ {E_stokes/0.065:.1f}")
print("(Lower S = narrower ZPL = better for quantum optics)")


## Platform 2: Boron Vacancy in hBN

Hexagonal boron nitride (hBN) is a layered 2D material with a wide bandgap (~6 eV). It hosts a variety of point defect SPEs, of which the **boron vacancy** (V_B) has attracted intense interest due to its room-temperature spin coherence.

### Why hBN?

- 2D geometry enables **near-field coupling** to photonic structures (no waveguide needed)
- **Ultra-bright** SPEs: count rates >10⁷ counts/s at room temperature
- Compatible with van der Waals heterostructures

### V_B electronic structure

The V_B has $D_{3h}$ symmetry with a spin-triplet ground state. Its ZPL is around **800 nm**, though the emission is broad (large Huang-Rhys factor).

### Key papers to read:
- Gottscholl et al., *Nature Materials* (2020) — room-temperature spin coherence in V_B
- Haykal et al., *Nature Communications* (2022) — spectroscopy of V_B centres


In [ ]:
from ase.build import bulk
from ase import Atoms
import numpy as np
import matplotlib.pyplot as plt

# Build monolayer hBN structure
# Lattice: hexagonal, a = 2.504 Å
a = 2.504
hbn = Atoms('BN',
            positions=[[0, 0, 0],
                       [a/2, a*np.sqrt(3)/6, 0]],
            cell=[[a, 0, 0],
                  [a/2, a*np.sqrt(3)/2, 0],
                  [0, 0, 20.0]],   # 20 Å vacuum
            pbc=[True, True, False])

print(f"hBN unit cell: {hbn.get_chemical_formula()}")
print(f"B-N bond length: {hbn.get_distance(0, 1):.4f} Å")
print(f"  (experimental: 1.446 Å)")

# Build 4×4 supercell for defect calculation
from ase.build import make_supercell
T = np.array([[4, 0, 0], [0, 4, 0], [0, 0, 1]])
hbn_super = make_supercell(hbn, T)
print(f"\n4×4 supercell: {len(hbn_super)} atoms, {hbn_super.get_chemical_formula()}")

# Create V_B: remove one boron atom
B_indices = [i for i, s in enumerate(hbn_super.get_chemical_symbols()) if s == 'B']
del hbn_super[B_indices[0]]
print(f"After V_B: {hbn_super.get_chemical_formula()}, {len(hbn_super)} atoms")


In [ ]:
# Visualise the hBN supercell with V_B defect
from ase.visualize.plot import plot_atoms

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_atoms(hbn_super, axes[0], rotation=('0x,0y,0z'), radii=0.4)
axes[0].set_title('hBN 4×4 supercell with V_B defect
(top view)', fontsize=12)
axes[0].axis('off')

# Show the local environment
local_env = hbn_super[:20].copy()   # first 20 atoms near defect
plot_atoms(local_env, axes[1], rotation=('0x,0y,0z'), radii=0.6)
axes[1].set_title('Local environment near V_B
(N atoms in teal, remaining B in pink)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Boron vacancy in hexagonal boron nitride', fontsize=13)
plt.tight_layout()
plt.show()


## Platform 3: Rare-Earth Ions in Wide-Gap Hosts

**Rare-earth (RE) ions** (Er³⁺, Nd³⁺, Yb³⁺, ...) have partially filled 4f shells shielded by outer 5s5p electrons. This shielding gives them extremely sharp optical transitions — ideal for quantum memories and telecom-compatible emitters.

### Key features:
- **Erbium (Er³⁺)**: emission at **1535 nm** — exact telecom C-band wavelength!
- Linewidths down to sub-MHz (lifetime-limited)
- Weak coupling to phonons (4f states are shielded)

### Common hosts:
- SiO₂ (optical fibres)
- Y₂SiO₅ (YSO) — excellent spin coherence for Er³⁺
- CaWO₄ — spin-photon interface
- Si — CMOS-compatible but challenged by phonon bath


In [ ]:
# Schematic energy level diagram for Er³⁺ (⁴I manifold)
fig, ax = plt.subplots(figsize=(5, 8))

levels = {
    '$^4I_{15/2}$ (ground)': 0,
    '$^4I_{13/2}$': 1.27,    # 1535 nm
    '$^4I_{11/2}$': 1.88,    # 660 nm pump
    '$^4I_{9/2}$':  2.52,
    '$^4F_{9/2}$':  3.07,
}

for label, E in levels.items():
    ax.plot([0.1, 0.9], [E, E], 'b-', lw=3)
    ax.text(0.95, E, label, va='center', fontsize=11)

# Main telecom transition
ax.annotate('', xy=(0.5, 0), xytext=(0.5, 1.27),
            arrowprops=dict(arrowstyle='->', color='red', lw=3))
ax.text(0.52, 0.6, '1535 nm
(telecom C-band)', color='red', fontsize=11)

ax.set_xlim(0, 2.5)
ax.set_ylim(-0.3, 3.5)
ax.set_ylabel('Energy (eV)', fontsize=12)
ax.set_title('Er³⁺ energy levels
(4f shell transitions)', fontsize=12)
ax.set_xticks([])
ax.spines[['right', 'top', 'bottom']].set_visible(False)
plt.tight_layout()
plt.show()


## Platform 4: Quantum Dots

**Quantum dots (QDs)** are nanoscale semiconductor inclusions where quantum confinement discretises the energy levels. They are bright, tunable SPEs.

### InAs/GaAs quantum dots:
- Self-assembled via Stranski-Krastanov growth
- ZPL at ~900–1000 nm, tunable by size
- Photon indistinguishability >99% at cryogenic temperatures
- Coupled to photonic crystal nanocavities for Purcell enhancement

### Perovskite quantum dots:
- CsPbX₃ (X = Cl, Br, I): tunable emission across the entire visible range
- High quantum yield (>90%)
- Solution processable — low cost
- Challenge: blinking, stability


In [ ]:
# Quantum confinement: particle-in-a-box model
# Demonstrates why QD emission wavelength depends on size

hbar = 1.054571817e-34  # J s
me   = 9.10938e-31       # kg
eV   = 1.60218e-19       # J
nm   = 1e-9

# InAs effective mass parameters
m_e_eff = 0.023 * me   # electron effective mass
m_h_eff = 0.36  * me   # hole effective mass
E_bulk_gap = 0.354      # eV (bulk InAs at 4 K)

# Confinement energy (simplified spherical model)
def confinement_energy(R_nm):
    R = R_nm * nm
    mu = m_e_eff * m_h_eff / (m_e_eff + m_h_eff)   # reduced mass
    E_conf = (hbar * np.pi)**2 / (2 * mu * R**2) / eV   # eV
    return E_conf

R_values = np.linspace(2, 20, 100)  # nm
E_conf = [confinement_energy(R) for R in R_values]
E_total = E_bulk_gap + np.array(E_conf)
wavelength = 1240 / E_total  # nm

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(R_values, E_total, 'crimson', lw=2)
axes[0].axhline(E_bulk_gap, color='gray', ls='--', label='Bulk InAs gap')
axes[0].set_xlabel('QD radius (nm)'); axes[0].set_ylabel('Emission energy (eV)')
axes[0].set_title('InAs QD emission energy vs size')
axes[0].legend()

axes[1].plot(R_values, wavelength, 'steelblue', lw=2)
axes[1].axhspan(900, 1700, alpha=0.15, color='green', label='Telecom window')
axes[1].set_xlabel('QD radius (nm)'); axes[1].set_ylabel('Emission wavelength (nm)')
axes[1].set_title('InAs QD emission wavelength vs size')
axes[1].legend()

plt.suptitle('Quantum confinement in InAs quantum dots', fontsize=13)
plt.tight_layout()
plt.show()

print("At R = 5 nm:")
print(f"  Emission energy: {E_bulk_gap + confinement_energy(5):.3f} eV")
print(f"  Emission wavelength: {1240/(E_bulk_gap + confinement_energy(5)):.0f} nm")


## Computational Workflow Summary

Here is the complete DFT workflow for characterising a solid-state quantum emitter:

```{figure} ../images/defect_workflow.png
:name: defect-workflow
:width: 85%
Complete computational workflow for characterising a point defect quantum emitter.
```

1. **Choose host and build supercell** (Lectures 2, 4)
   - Wide-bandgap semiconductor or insulator
   - Supercell large enough for isolated defect (~200 atoms)

2. **Create the defect** (Lecture 4)
   - Substitution, vacancy, or complex
   - Set the correct charge state

3. **Relax geometry** (Lecture 8)
   - Ground state: relax fully
   - Excited state: relax with constrained occupation (Δ-SCF)

4. **Compute electronic structure** (Lecture 6)
   - Use HSE06 or SCAN for accurate gap and level positions
   - Identify defect levels in the gap

5. **Compute optical properties**
   - ZPL = E(excited, relaxed) − E(ground, relaxed)
   - Huang-Rhys factor from mass-weighted displacements
   - Radiative lifetime from transition dipole moment

6. **Compute spin properties** (for spin qubits)
   - Zero-field splitting from spin-orbit + spin-spin coupling
   - Hyperfine constants

---

## Key Points

- Solid-state SPEs span diamond (NV), hBN (V_B), QDs (InAs, perovskite), and RE ions
- Key figures of merit: ZPL fraction, linewidth, emission wavelength, brightness
- DFT workflow: build → relax → electronic structure → optical properties
- HSE06 (or G₀W₀) is needed for accurate bandgaps and defect level positions
- The configuration coordinate diagram connects DFT geometry to optical spectra

## Exercise 9.1

Using the quantum confinement model above, find the InAs QD radius that gives emission at the telecom O-band (1310 nm). Repeat for the telecom C-band (1550 nm).

## Exercise 9.2

The Huang-Rhys factor $S$ for the NV centre is approximately 3.67. For V_B in hBN it is approximately 4 (large phonon sideband). What does this imply for the ZPL fraction in each case? Use the relation ZPL fraction ≈ $e^{-S}$. Which is better for quantum optics and why?

## Exercise 9.3 (Research project starter)

Choose one of the following emerging quantum emitter platforms and write a 1-page summary of its key properties, advantages, and open research questions:
- Silicon carbide (SiC) divacancy
- Boron nitride nanotube defects  
- Europium (Eu³⁺) in Y₂O₃
- Defects in 2D transition metal dichalcogenides (TMDs)

Use the computational criteria from this lecture to frame your discussion.
